# 20 — BERTopic Cluster Parameter Search

Samples **10,000 reviews per language** and sweeps `min_cluster_size` × `min_samples`
to find appropriate parameters before the full coast-band re-fit.

### What to look for

Raw HDBSCAN outlier % (no `reduce_outliers` applied):

- Too high (> 50%) → `min_cluster_size` is too large — rejecting natural small clusters as noise
- Too low (< 15%) → `min_cluster_size` is too small — forming noisy micro-clusters
- Sweet spot: **15–35% for EN**, **20–40% for VI**

Among combos in the sweet spot, pick the one with the **most raw topics** (finer granularity).

In [1]:
import sys
sys.path.insert(0, "..")

import random
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from stopwordsiso import stopwords
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from bertopic.vectorizers import ClassTfidfTransformer

from src.topic_modeling import load_from_duckdb

DB_PATH     = Path("../data/hotel_reviews.db")
SAMPLE_SIZE = 10_000
RANDOM_SEED = 42

encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
print("Encoder loaded.")

c:\Users\darkn\coastal-hotel\.venv\Lib\site-packages\stopwordsiso\_core.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoder loaded.


## Section 1 — Sample 10k Reviews per Language

In [2]:
samples = {}

for lang in ["en", "vi"]:
    df_all, docs_all, emb_all = load_from_duckdb(db_path=DB_PATH, language=lang)
    print(f"{lang}: {len(docs_all):,} total reviews")

    random.seed(RANDOM_SEED)
    idx = sorted(random.sample(range(len(docs_all)), min(SAMPLE_SIZE, len(docs_all))))

    samples[lang] = {
        "docs":       [docs_all[i] for i in idx],
        "embeddings": emb_all[idx],
    }
    print(f"  Sampled: {len(samples[lang]['docs']):,}  emb shape: {samples[lang]['embeddings'].shape}")

print("\nSamples ready.")

[load_from_duckdb] Loading 134,572 rows …
[load_from_duckdb] docs: 134,572  embeddings: (134572, 768)
en: 134,572 total reviews
  Sampled: 10,000  emb shape: (10000, 768)
[load_from_duckdb] Loading 116,756 rows …
[load_from_duckdb] docs: 116,756  embeddings: (116756, 768)
vi: 116,756 total reviews
  Sampled: 10,000  emb shape: (10000, 768)

Samples ready.


## Section 2 — Parameter Grid Search

Grid: `min_cluster_size` × `min_samples` × language

In [ ]:
import math

# Rule: min_samples = ceil(min_cluster_size / 2)
MIN_CLUSTER_SIZES = [5, 10, 15, 20, 25]

def auto_min_samples(mcs: int) -> int:
    return math.ceil(mcs / 2)

results = []

for lang in ["en", "vi"]:
    docs       = samples[lang]["docs"]
    embeddings = samples[lang]["embeddings"]

    for mcs in MIN_CLUSTER_SIZES:
        ms = auto_min_samples(mcs)

        print(f"  [{lang}] min_cluster_size={mcs}  min_samples={ms} ...", end=" ", flush=True)

        topic_model = BERTopic(
            embedding_model=encoder,
            umap_model=UMAP(
                n_neighbors=10,
                n_components=5,
                min_dist=0.0,
                metric="cosine",
                random_state=RANDOM_SEED,
            ),
            hdbscan_model=HDBSCAN(
                min_cluster_size=mcs,
                min_samples=ms,
                metric="euclidean",
                cluster_selection_method="eom",
                prediction_data=False,
            ),
            vectorizer_model=CountVectorizer(
                stop_words=list(stopwords(["vi", "en"])),
                min_df=2,
                ngram_range=(1, 2),
            ),
            ctfidf_model=ClassTfidfTransformer(),
            representation_model={
                "KeyBERT": KeyBERTInspired(),
                "MMR":     MaximalMarginalRelevance(diversity=0.3),
            },
            nr_topics="auto",
            min_topic_size=mcs,
            top_n_words=20,
            calculate_probabilities=False,
            verbose=False,
        )

        topics, _ = topic_model.fit_transform(docs, embeddings)

        n_topics  = len(set(topics)) - (1 if -1 in topics else 0)
        n_outlier = sum(1 for t in topics if t == -1)
        pct       = n_outlier / len(topics) * 100

        print(f"topics={n_topics}  outliers={n_outlier:,} ({pct:.1f}%)")

        results.append({
            "lang":             lang,
            "min_cluster_size": mcs,
            "min_samples":      ms,
            "n_topics":         n_topics,
            "n_outliers":       n_outlier,
            "outlier_pct":      round(pct, 1),
        })

print("\nGrid search done.")

## Section 3 — Results Table

Sweet spot per language (raw outlier % only):
- **EN**: outlier 15–35%, most topics wins
- **VI**: outlier 20–40%, most topics wins

In [ ]:
df_results = pd.DataFrame(results)

SWEET = {
    "en": {"lo": 15, "hi": 35},
    "vi": {"lo": 20, "hi": 40},
}

def row_style(row):
    t = SWEET[row["lang"]]
    if t["lo"] <= row["outlier_pct"] <= t["hi"]:
        color = "#d4edda"   # green — sweet spot
    elif row["outlier_pct"] <= t["hi"] + 10:
        color = "#fff3cd"   # yellow — acceptable
    else:
        color = ""
    return [f"background-color: {color}" for _ in row]

from IPython.display import display

for lang in ["en", "vi"]:
    sub = df_results[df_results["lang"] == lang].reset_index(drop=True)
    t   = SWEET[lang]
    print(f"\n=== {lang.upper()} — sweet spot: outlier {t['lo']}–{t['hi']}% ===")
    display(
        sub.style
        .apply(row_style, axis=1)
        .format({"outlier_pct": "{:.1f}%"})
    )

## Section 4 — Top 3 Candidates per Language

Ranked by: outlier % in sweet spot first, then most topics (finer granularity).

In [ ]:
for lang in ["en", "vi"]:
    t   = SWEET[lang]
    sub = df_results[df_results["lang"] == lang].copy()

    # Prefer rows in sweet spot, then fewest outliers, then most topics
    sub["in_sweet"] = sub["outlier_pct"].between(t["lo"], t["hi"]).astype(int)
    top3 = (
        sub.sort_values(["in_sweet", "outlier_pct", "n_topics"],
                        ascending=[False, True, False])
        .head(3)
        .reset_index(drop=True)
    )
    print(f"\nTop 3 for [{lang.upper()}]:")
    print(top3[["min_cluster_size", "min_samples", "n_topics", "n_outliers", "outlier_pct"]].to_string(index=False))